# Interactive Dashboard Report

## Introduction

*<In two or three sentences: state the question or story your dashboard explores — what can a reader discover with it that they couldn't from a single static chart?>*

**Finished dashboard**

*<Insert a screenshot of the finished dashboard here, and add the link to the live version below.>*

*Live version:* https://siads-521-energy-dashboard-7474654444631410.aws.databricksapps.com/

In [0]:
#%pip install dash
import pandas as pd
from dash import dcc, html
import plotly.express as px
import plotly.graph_objects as go

## Library choice and fit (15 pts)

**The library: Dash**

- **Created by:** Plotly (the company behind Plotly.js and Plotly Express)
- **License:** MIT — a permissive license: free for any use including commercial, and it may be modified and redistributed; the only requirement is preserving the copyright and license notice.
- **How to install:** `%pip install dash`
- **Declarative or procedural:** declarative. The UI is described as a tree of components — `html.Div` containers, `dcc.Graph` figures, `dcc.Dropdown` / `dcc.RangeSlider` controls — and behavior is declared as callback mappings (Inputs → Outputs).

**Jupyter integration and limitations**

- Jupyter Integration: Dash runs by starting its dev server hosted locally (http://127.0.0.1:8050) opening via browswer.
- Limitations hit in this project: Notebook was developed on databricks so once getting to the testing of the dash app, was unable to run and test via local version.  Either had to commit changes to git, sync locally and test or fully deploy and test.  Both took a decent amount of time.

**Why Dash fits this dataset**

The dataset is tidy and structured — one row per country-year with a fixed set of metrics — and the questions are comparative and filter-driven: which countries, which years, and what changes together. That maps directly onto Dash's model: a single country dropdown and a single year-range slider drive all four views through one callback, so every chart always reflects the same selection. The choropleth depends on hover tooltips and the slope chart is rebuilt for whatever two endpoints the reader picks — interactivity that only a web framework provides, not a static plot.

**Alternative considered and rejected:** Streamlit. Its model reruns the whole script top-to-bottom on every widget change, so each slider tweak would redo all data prep and rebuild all four figures; Dash's targeted Input→Output callbacks let me update only what actually depends on the changed input (the map uses the range's end year, the slope chart its two endpoints, while the line and area charts redraw over the full range). Dash also reuses my Plotly Express figures directly, with no conversion layer.

## Data

- **Source:** the Our World in Data **Energy** dataset (`owid_energy.csv`), which OWID compiles from the Energy Institute *Statistical Review of World Energy* and related sources — https://ourworldindata.org/energy
- **License:** CC BY 4.0 (Creative Commons Attribution) — the dataset can be shared, adapted, and built upon, including commercially, as long as Our World in Data is credited.
- **Size (raw):** 19,494 rows × 130 columns (9.1 MB)
- **Size (cleaned):** 8,786 rows × 17 columns — 221 countries/regions (including the "World" aggregate) spanning 1985–2025; years before 1985 are dropped because values are largely missing

**Columns used**

- `country` — name of the country or aggregate region (string)
- `iso_code` — ISO country code; missing for aggregates such as "World" (string)
- `year` — calendar year of the observation (integer)
- `bioenergy_electricity_twh`, `coal_electricity_twh`, `gas_electricity_twh`, `hydro_electricity_twh`, `nuclear_electricity_twh`, `oil_electricity_twh`, `solar_electricity_twh`, `wind_electricity_twh`, `other_renewables_excluding_bioenergy_electricity_twh` — electricity generation by source, in terawatt-hours (float)
- `renewables_electricity_share_pct` — share of electricity generated from renewables, % (float)
- `fossil_fuels_electricity_share_pct` — share of electricity generated from fossil fuels, % (float)
- `electricity_generation_twh` — total electricity generation in terawatt-hours; used to rank the dashboard's default selection of countries (float)
- `electricity_generation_per_capita_kwh` — electricity generated per person, kWh; plotted as "energy generation" (float)
- `electricity_emissions_mtco2eq` — greenhouse-gas emissions from electricity generation, MtCO2eq (float)

### Cleaning steps
Every cleaning step, in code, with a sentence explaining each.

In [0]:
#data import and cleaning
df = pd.read_csv('owid_energy.csv')

#define columns to be used
columns = [
    'country',
    'iso_code',
    'year',
    'bioenergy_electricity_twh',
    'coal_electricity_twh',
    'gas_electricity_twh',
    'hydro_electricity_twh',
    'nuclear_electricity_twh',
    'oil_electricity_twh',
    'solar_electricity_twh',
    'wind_electricity_twh',
    'other_renewables_excluding_bioenergy_electricity_twh',
    'renewables_electricity_share_pct',
    'fossil_fuels_electricity_share_pct',
    'electricity_generation_twh',
    'electricity_generation_per_capita_kwh',
    'electricity_emissions_mtco2eq',
]

#get list of all countries by name with no iso code, but add back in world
countries = df[df['iso_code'].notna()]['country'].unique().tolist()
countries.append('World')

#data filtering
#Data prior to 1985 is missing values, 
#remove any countries that have no ISO code
mask = (
    (df['year'] >= 1985) &
    (df['country'].isin(countries))
    )

df = df[mask][columns]

## Visualization techniques (20 pts)

- **Line chart (fig1)** — shows how one continuous metric (energy generation per capita) evolves over time for each selected country. Used here because the first question is a time-trend comparison across a handful of countries: one line per country makes levels, slopes, and crossovers directly comparable, and the unified hover reads values at the same year across all lines. Avoid when there are many series (spaghetti), the x-axis is not an ordered continuous quantity, or the question is comparing categories at one point in time — bars serve that better.

- **Stacked area chart (fig2)** — shows the total generation and its composition by source over time: the whole height is the total, each band a source. Used here because the story is "growing total, changing mix" — the coal → gas → renewables shift is visible in how the bands expand and shrink, and the legend is ordered by the largest source in the final plotted year so the biggest contributors read first. Avoid when many categories are tiny (unreadable slivers), when individual series trends matter more than the part-to-whole view (stacking makes each band's shape depend on everything stacked beneath it), or when values can be negative.

- **Choropleth (fig3)** — shows a single metric by geography; color encodes the share of generation from renewables, 0–100% on a fixed RdYlGn scale. Used here because "where has the transition happened?" is inherently spatial — the map exposes regional clustering (e.g. Scandinavia and Latin America vs fossil-rich regions) that a sorted bar list hides, and the fixed 0–100 range means the same color means the same share regardless of the selected year. Avoid when precise comparisons are needed (the eye compares shades poorly), when large countries visually dominate despite small values, or when the metric is a raw total rather than a rate — area and color can mislead.

**New-to-class method: slope chart (fig5)**

A slope chart is a line chart reduced to exactly two time points: two vertical axes — the first and last year of the selected range — with one line per country connecting its renewables share at each endpoint, the country name labeled directly at the line's right end instead of in a legend. What it reveals that a class-covered chart would not: it distills the whole selected range into the *net change* per country, so direction and steepness rank the transition at a glance — who surged, who stalled, who slipped back. A full multi-series line chart like fig1 shows the whole trajectory, but the endpoint-to-endpoint comparison gets buried in everything in between; the slope chart answers exactly "how much did each selected country's renewables share change between these two years?" (An earlier parallel-coordinates figure, fig4, is kept in the Step 1 components as a static exploration, but the slope chart replaced it in the dashboard.) Caveats: it only works for exactly two time points — the year-range control guarantees this by always using the range's endpoints — a long country list turns it into overlapping spaghetti (the dropdown's default selection of ten keeps it readable), and because the underlying pivot fills missing endpoint values with zero, a country with no data at one endpoint reads as 0% there, so a steep line can indicate a data gap rather than a real change.

**How the charts complement each other**

The layout follows the reader's question flow. Row 1 (line + area) answers *how much* and *from what* over the selected years, for the chosen countries; row 2 pairs the map with the slope chart, putting *where the transition stands* (at the end of the selected range, across every country) side by side with *how much each selected country's renewables share changed* between the range's endpoints. The shared country dropdown and year-range slider keep all four views consistent with one selection, so a reader can notice a country on the map, add it in the dropdown, and immediately read its trajectory, its mix, and its net change — filter, compare, then drill down.

## Building and exploring (35 pts)

Build the dashboard step by step: basic components first, then layout, then linked interactivity. Include a code snippet **and a screenshot** at each key stage.

### Step 1 — Basic components

*<Define individual figures / data preparation for the components. Code snippet + screenshot.>*

In [0]:
def generate_country_data(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1985,2026)) -> pd.DataFrame():
    dfc = df.copy()
    result = dfc[dfc['country'].isin(countries) & dfc['year'].isin(years)].rename(columns = {'electricity_generation_per_capita_kwh': 'energy generation'})

    return result

data = generate_country_data(df, ['United States', 'United Kingdom'])

# figure 1
fig1 = px.line(
    data,
    x = 'year',
    y = 'energy generation',
    color = 'country',
    markers = True,
    title = f'Total Energy Generation per Capita by Country ({min(data['year'])}-{max(data['year'])})',
    hover_data = {
        'year': False, 
        'country': True, 
        'energy generation': ":,.0f"},
    )

fig1.update_layout(
    plot_bgcolor = 'rgba(0,0,0,0)',
    paper_bgcolor = 'rgba(0,0,0,0)',
    xaxis_title = 'Year',
    yaxis_title = 'Total Energy (kWh)',
    legend_title ='Country',
    hovermode = 'x unified',
)

#black axis lines so the axes stay visible on the transparent background
fig1.update_xaxes(showline = True, linecolor = 'black', tickcolor = 'black')
fig1.update_yaxes(showline = True, linecolor = 'black', tickcolor = 'black')

fig1.show()

In [0]:
def generate_energy_share(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1985,2026)) -> pd.DataFrame():
    dfc = df.copy()

    #define and get columnms we want
    columns = [
        'country',
        'year',
        'bioenergy_electricity_twh',
        'coal_electricity_twh',
        'gas_electricity_twh',
        'hydro_electricity_twh',
        'nuclear_electricity_twh',
        'oil_electricity_twh',
        'solar_electricity_twh',
        'wind_electricity_twh',
        'other_renewables_excluding_bioenergy_electricity_twh',
        ]

    #create dataframe and remove suffixes and fix column names for display
    result = dfc[columns].rename(columns = lambda x: x.removesuffix('_electricity_twh')).rename(columns = {'other_renewables_excluding_bioenergy': 'other renewables'})

    #convert to long format
    result = pd.melt(result, id_vars = ['country', 'year'], var_name = 'energy_type', value_name = 'values')

    #filter to the selected countries and sum their values for each year and energy type
    result = (result[result['country'].isin(countries) & result['year'].isin(years)]
              .groupby(['year', 'energy_type'], as_index = False)['values'].sum())

    #create sort order based on values for final plot year
    order = result[result['year'] == df['year'].max()].sort_values('values', ascending = False)['energy_type'].to_list()

    return result, order

data, order = generate_energy_share(df)

#figure2
fig2 = px.area(
    data,
    x = 'year',
    y = 'values',
    color = 'energy_type',
    line_group = 'energy_type',
    category_orders = {'energy_type': order}, #order of plotting is based on values of right most year being plotted with largest at the bottom
    title = f'Total Energy Generated by Source ({min(data['year'])}-{max(data['year'])})',
    hover_data = {
        'year': False,
        'energy_type': True,
        'values': ":,.0f"
    }
)

fig2.update_layout(
    plot_bgcolor = 'rgba(0,0,0,0)',
    paper_bgcolor = 'rgba(0,0,0,0)',
    xaxis_title = 'Year',
    yaxis_title = 'Total Energy Generated (TWh)',
    legend_title ='Energy Source',
    hovermode = 'x unified'
)

#black axis lines so the axes stay visible on the transparent background
fig2.update_xaxes(showline = True, linecolor = 'black', tickcolor = 'black')
fig2.update_yaxes(showline = True, linecolor = 'black', tickcolor = 'black')

fig2.show()

In [0]:
def generate_renewable_energy(df: pd.DataFrame(), years: list = range(1985,2026)) -> pd.DataFrame():
    #copy dataframe and restrict to most recent year
    dfc = df[df['year'] == max(years)].copy()

    #drop columns and na values from iso_code and renewables_electricity_share_pct
    result = dfc[['country','iso_code','year', 'renewables_electricity_share_pct']].dropna(subset = ['iso_code', 'renewables_electricity_share_pct']).rename(columns = {'renewables_electricity_share_pct': 'renewables_share'})

    return result

data = generate_renewable_energy(df)

#figure 3
fig3 = px.choropleth(
    data,
    locations = 'iso_code',
    locationmode = 'ISO-3',
    color = 'renewables_share',
    hover_name = 'country',
    hover_data = {
        'iso_code': False,
        'year': True,
        'renewables_share': ":.0f"
    },
    color_continuous_scale = 'RdYlGn',
    range_color = [0, 100],
    title = f'Renewable Energy Generation by Country ({max(data['year'])})'
)

fig3.update_layout(
    title_x=0.5,
    height = 650,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    coloraxis_colorbar=dict(
        title="Renewable<br>Generation",
        ticksuffix="%"
    )
)

fig3.show()

In [0]:
def generate_profiles(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1985,2026)) -> pd.DataFrame():
    dfc = df.copy()

    #define columns required
    columns = [
        'country',
        'year',
        'electricity_generation_per_capita_kwh',
        'renewables_electricity_share_pct',
        'fossil_fuels_electricity_share_pct',
        'electricity_emissions_mtco2eq',
    ]

    #select required columns and rename
    result = (dfc[columns].rename(columns = {
        'electricity_generation_per_capita_kwh': 'energy_per_capita', 
        'renewables_electricity_share_pct': 'renewables_share', 
        'fossil_fuels_electricity_share_pct': 'fossil_fuels_share', 
        'electricity_emissions_mtco2eq': 'emissions'}))
    
    #filter to countries and years, drop any that have na
    result = (result[(result['country'].isin(countries)) & (result['year'] == max(years))]
              .dropna(subset = ['energy_per_capita','renewables_share', 'fossil_fuels_share', 'emissions']))

    return result

#test values
data = generate_profiles(df, countries = ['United States', 'China','United Kingdom'])

#figure 4
fig4 = px.parallel_coordinates(
    data,
    dimensions = [
        'energy_per_capita',
        'renewables_share',
        'fossil_fuels_share',
        'emissions'
    ],
    color = 'renewables_share',
    color_continuous_scale = 'RdYlGn',
    range_color = [0, 100],
    labels = {
        'energy_per_capita': 'Energy per capita (kWh)',
        'renewables_share': 'Renewables share (%)',
        'fossil_fuels_share': 'Fossil fuels share (%)',
        'emissions': 'Emissions (MtCO2eq)'
    },
    title = f'Energy Profiles of Countries ({max(data['year'])})'
)

fig4.update_layout(
    title_x=0.5,
    height=650,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    coloraxis_colorbar=dict(
        title="Renewable<br>Share",
        ticksuffix="%"
    )
)

fig4.show()

In [0]:
def generated_differential(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1985,2026)) -> pd.DataFrame():
    dfc = df.copy()

    #define columns required
    columns = [
        'country',
        'year',
        'renewables_electricity_share_pct'
        ]
    
    #filter for countries and years
    result = (dfc[columns][(dfc['country'].isin(countries)) & (dfc['year'].isin([min(years), max(years)]))])

    #pivot to get min and max on separate columns, fill null with 0
    result = result.pivot(index = 'country', columns = 'year', values = 'renewables_electricity_share_pct').reset_index().fillna(0)

    return result

data = generated_differential(df, countries = ['United States', 'China', 'Canada', 'Brazil', 'France' ])

#dodge overlapping labels — plotly has no built-in label repulsion, so cluster
#close endpoints and spread their labels to keep a minimum vertical gap
gap = 4 #minimum label separation in %-points (~19px at this chart height)
end_col = data.columns[2]
pts = data[['country', end_col]].sort_values(end_col, ascending = False).reset_index(drop = True)
label_y = {}
i = 0
while i < len(pts):
    #grow the cluster while its span is too tight to fit all labels with gaps
    j = i
    while j + 1 < len(pts) and pts.loc[i, end_col] - pts.loc[j + 1, end_col] < gap * (j + 1 - i):
        j += 1
    cluster = pts.loc[i:j]
    #spread the cluster's labels evenly, centered on the cluster's midpoint
    mid = (pts.loc[i, end_col] + pts.loc[j, end_col]) / 2
    top = min(100, mid + gap * (len(cluster) - 1) / 2)
    top = max(top, gap * (len(cluster) - 1)) #keep the spread inside the 0-100 axis range
    for k, (_, r) in enumerate(cluster.iterrows()):
        label_y[r['country']] = top - k * gap
    i = j + 1

#figure 4.2
fig5 = go.Figure()

for _, row in data.iterrows():
    fig5.add_trace(
        go.Scatter(
        x = [0, 1],
        y = [row[data.columns[1]], row[data.columns[2]]],
        mode = 'lines+markers',
        name = row['country'],
        cliponaxis = False,
        )
    )

#country labels at the (possibly nudged) right endpoints — annotations render
#in the margin layer, take a pixel offset, and hover ignores them
for _, row in data.iterrows():
    fig5.add_annotation(
        x = 1,
        y = label_y[row['country']],
        xref = 'x',
        yref = 'y',
        text = row['country'],
        showarrow = False,
        xanchor = 'left',
        xshift = 8, #nudge labels slightly right, away from the axis line
        font = dict(color = 'black')
    )

#vertical line on the right side — the right axis of the slope chart
fig5.add_shape(
    type = 'line',
    xref = 'x', yref = 'paper',
    x0 = 1, y0 = 0,
    x1 = 1, y1 = 1,
    line = dict(color = 'black', width = 1)
)

fig5.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    title=(
        f"Change in Renewable Energy Share "
        f"({data.columns[1]}–{data.columns[2]})"
    ),

    title_x=0.5,
    height=650,
    showlegend=False,
    margin=dict(
        l=80,
        r=160,
        t=100,
        b=70
    ),
    xaxis = dict(
        tickmode = 'array',
        tickvals = [0, 1],
        ticktext = [data.columns[1], data.columns[2]],
        showgrid = False,
        showline = False,
        ticks = 'outside',
        ticklen = 15,
        tickcolor = 'rgba(0,0,0,0)', #invisible tick marks push the year labels away from the axis
        range = [0, 1]
    ),
    yaxis = dict(
        title = 'Renewable Energy Share (%)',
        range = [0, 100],
        ticksuffix = '%',
        dtick = 20,
        showline = True,
        linecolor = 'black',
        tickcolor = 'black',
    )
)

fig5.show()



### Step 2 — Layout

*<Assemble components into the app layout. Code snippet + screenshot of the first full render.>*


In [0]:
# Step 2 — Layout: assemble the four figures into the Dash app
from dash import Dash, dcc, html
from threading import Thread

app = Dash(__name__)

# Row 1 — fig1 (energy per capita) and fig2 (generation by source) side by side
row1 = html.Div(
    [
        dcc.Graph(figure=fig1, style={'flex': 1}),
        dcc.Graph(figure=fig2, style={'flex': 1}),
    ],
    style={'display': 'flex', 'gap': '10px'},
)

# Row 2 — fig3 (renewables choropleth) and fig5 (change in renewables share) side by side
row2 = html.Div(
    [
        dcc.Graph(figure=fig3, style={'flex': 1}),
        dcc.Graph(figure=fig5, style={'flex': 1}),
    ],
    style={'display': 'flex', 'gap': '10px'},
)

app.layout = html.Div([row1, row2])

# Serve the app on a background thread so the notebook stays responsive
Thread(target=lambda: app.run(host='0.0.0.0', port=8050), daemon=True).start()
print('Dashboard serving at http://127.0.0.1:8050')

### Step 3 — Linked interactivity

*<Add callbacks that connect filters/charts. Code snippet + screenshot showing the interaction (before/after states).>*


In [0]:
# Step 3 — Linked interactivity: shared country/year controls drive all four figures
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
from threading import Thread
import plotly.graph_objects as go

def empty_figure(title, message):
    fig = go.Figure()
    fig.update_layout(
        title = title,
        title_x = 0.5,
        annotations = [dict(text = message, showarrow = False, x = 0.5, y = 0.5,
                            xref = 'paper', yref = 'paper', font = dict(size = 16))]
    )
    return fig

# Figure builders — the Step 1 chart code, parameterized by the shared controls
def make_fig1(countries, years):
    data = generate_country_data(df, countries = countries, years = years)
    if data.empty:
        return empty_figure('Total Energy Generation per Capita', 'No data for the selected countries')
    fig = px.line(
        data,
        x = 'year',
        y = 'energy generation',
        color = 'country',
        markers = True,
        title = f"Total Energy Generation per Capita by Country ({min(data['year'])}-{max(data['year'])})",
        hover_data = {
            'year': False,
            'country': True,
            'energy generation': ':,.0f'
        },
    )
    fig.update_layout(
        plot_bgcolor = 'rgba(0,0,0,0)',
        paper_bgcolor = 'rgba(0,0,0,0)',
        xaxis_title = 'Year',
        yaxis_title = 'Total Energy (kWh)',
        legend_title = 'Country',
        hovermode = 'x unified'
    )
    #black axis lines so the axes stay visible on the transparent background
    fig.update_xaxes(showline = True, linecolor = 'black', tickcolor = 'black')
    fig.update_yaxes(showline = True, linecolor = 'black', tickcolor = 'black')
    return fig

def make_fig2(countries, years):
    data, order = generate_energy_share(df, countries = countries, years = years)
    if data.empty:
        return empty_figure('Energy Generation by Source', 'No data for the selected countries')
    # order the stack by the right-most year actually being plotted
    order = data[data['year'] == max(years)].sort_values('values', ascending = False)['energy_type'].to_list()
    fig = px.area(
        data,
        x = 'year',
        y = 'values',
        color = 'energy_type',
        line_group = 'energy_type',
        category_orders = {'energy_type': order},
        title = f"Total Energy Generated by Source ({min(data['year'])}-{max(data['year'])})",
        hover_data = {
            'year': False,
            'energy_type': True,
            'values': ':,.0f'
        },
    )
    fig.update_layout(
        plot_bgcolor = 'rgba(0,0,0,0)',
        paper_bgcolor = 'rgba(0,0,0,0)',
        xaxis_title = 'Year',
        yaxis_title = 'Total Energy Generated (TWh)',
        legend_title = 'Energy Source',
        hovermode = 'x unified'
    )
    #black axis lines so the axes stay visible on the transparent background
    fig.update_xaxes(showline = True, linecolor = 'black', tickcolor = 'black')
    fig.update_yaxes(showline = True, linecolor = 'black', tickcolor = 'black')
    return fig

def make_fig3(year):
    data = generate_renewable_energy(df, years = [year])
    if data.empty:
        return empty_figure(f'Renewable Energy Generation ({year})', 'No data for this year')
    fig = px.choropleth(
        data,
        locations = 'iso_code',
        locationmode = 'ISO-3',
        color = 'renewables_share',
        hover_name = 'country',
        hover_data = {'iso_code': False, 'year': True, 'renewables_share': ':.0f'},
        color_continuous_scale = 'RdYlGn',
        range_color = [0, 100],
        title = f'Renewable Energy Generation by Country ({year})'
    )
    fig.update_layout(
        title_x = 0.5,
        height = 650,
        plot_bgcolor = 'rgba(0,0,0,0)',
        paper_bgcolor = 'rgba(0,0,0,0)',
        coloraxis_colorbar = dict(title = 'Renewable<br>Generation', ticksuffix = '%')
    )
    return fig

def make_fig5(countries, years):
    data = generated_differential(df, countries = countries, years = years)
    if data.empty or len(data.columns) < 3:
        return empty_figure(f'Change in Renewable Energy Share ({min(years)}–{max(years)})', 'No data for this selection')
    start_year, end_year = data.columns[1], data.columns[2]
    #dodge overlapping labels — plotly has no built-in label repulsion, so cluster
    #close endpoints and spread their labels to keep a minimum vertical gap
    gap = 4 #minimum label separation in %-points (~19px at this chart height)
    pts = data[['country', end_year]].sort_values(end_year, ascending = False).reset_index(drop = True)
    label_y = {}
    i = 0
    while i < len(pts):
        #grow the cluster while its span is too tight to fit all labels with gaps
        j = i
        while j + 1 < len(pts) and pts.loc[i, end_year] - pts.loc[j + 1, end_year] < gap * (j + 1 - i):
            j += 1
        cluster = pts.loc[i:j]
        #spread the cluster's labels evenly, centered on the cluster's midpoint
        mid = (pts.loc[i, end_year] + pts.loc[j, end_year]) / 2
        top = min(100, mid + gap * (len(cluster) - 1) / 2)
        top = max(top, gap * (len(cluster) - 1)) #keep the spread inside the 0-100 axis range
        for k, (_, r) in enumerate(cluster.iterrows()):
            label_y[r['country']] = top - k * gap
        i = j + 1
    fig = go.Figure()
    for _, row in data.iterrows():
        fig.add_trace(
            go.Scatter(
                x = [0, 1],
                y = [row[start_year], row[end_year]],
                mode = 'lines+markers',
                name = row['country'],
                cliponaxis = False,
            )
        )
    #country labels at the (possibly nudged) right endpoints — annotations render
    #in the margin layer, take a pixel offset, and hover ignores them
    for _, row in data.iterrows():
        fig.add_annotation(
            x = 1,
            y = label_y[row['country']],
            xref = 'x',
            yref = 'y',
            text = row['country'],
            showarrow = False,
            xanchor = 'left',
            xshift = 8, #nudge labels slightly right, away from the axis line
            font = dict(color = 'black')
        )
    #vertical line on the right side — the right axis of the slope chart
    fig.add_shape(
        type = 'line',
        xref = 'x', yref = 'paper',
        x0 = 1, y0 = 0,
        x1 = 1, y1 = 1,
        line = dict(color = 'black', width = 1)
    )

    fig.update_layout(
        plot_bgcolor = 'rgba(0,0,0,0)',
        paper_bgcolor = 'rgba(0,0,0,0)',
        title = f'Change in Renewable Energy Share ({start_year}–{end_year})',
        title_x = 0.5,
        height = 650,
        showlegend = False,
        margin = dict(l = 80, r = 160, t = 100, b = 70),
        xaxis = dict(
            tickmode = 'array',
            tickvals = [0, 1],
            ticktext = [start_year, end_year],
            showgrid = False,
            showline = False,
            ticks = 'outside',
            ticklen = 15,
            tickcolor = 'rgba(0,0,0,0)', #invisible tick marks push the year labels away from the axis
            range = [0, 1]
        ),
        yaxis = dict(
            title = 'Renewable Energy Share (%)',
            range = [0, 100],
            ticksuffix = '%',
            dtick = 20,
            showline = True,
            linecolor = 'black',
            tickcolor = 'black'
        )
    )
    return fig

# Shared controls at the top of the dashboard
# Default selection: the top 10 countries by total electricity generation (TWh).
# The 'World' aggregate is excluded from the ranking — it isn't a country and
# would otherwise top the list (it stays selectable in the dropdown).
# The very latest year is often incomplete (2025 has values for only 91 of 221
# countries), so rank on the most recent year with at least 90% country coverage.
countries_only = df[df['country'] != 'World']
coverage = (countries_only.dropna(subset = ['electricity_generation_twh'])
              .groupby('year')['country'].nunique())
rank_year = coverage[coverage >= 0.9 * countries_only['country'].nunique()].index.max()
default_countries = (
    countries_only[countries_only['year'] == rank_year]
      .dropna(subset = ['electricity_generation_twh'])
      .nlargest(10, 'electricity_generation_twh')['country']
      .tolist()
)
country_options = sorted(df['country'].unique())
controls = html.Div(
    [
        html.Div(
            [
                html.Label('Countries', htmlFor = 'country-dropdown'),
                dcc.Dropdown(
                    id = 'country-dropdown',
                    options = country_options,
                    value = default_countries,
                    multi = True,
                    placeholder = 'Select countries…'
                ),
            ],
            style = {'flex': 3, 'paddingRight': '20px'}
        ),
        html.Div(
            [
                html.Label('Year range (the map uses the end year)', htmlFor = 'year-range'),
                dcc.RangeSlider(
                    id = 'year-range',
                    min = 1985,
                    max = int(df['year'].max()),
                    step = 1,
                    value = [1985, int(df['year'].max())],
                    allowCross = False,
                    marks = {y: str(y) for y in range(1985, int(df['year'].max()) + 1, 10)}
                ),
            ],
            style = {'flex': 2}
        ),
    ],
    style = {'display': 'flex', 'alignItems': 'flex-end', 'padding': '10px'}
)

app = Dash(__name__)
app.layout = html.Div(
    [
        controls,
        # Row 1 — fig1 and fig2 side by side
        html.Div(
            [
                dcc.Graph(id = 'fig1-graph', style = {'flex': 1}),
                dcc.Graph(id = 'fig2-graph', style = {'flex': 1}),
            ],
            style = {'display': 'flex', 'gap': '10px'}
        ),
        # Row 2 — fig3 (map) and fig5 (slope chart) side by side
        html.Div(
            [
                dcc.Graph(id = 'fig3-graph', style = {'flex': 1}),
                dcc.Graph(id = 'fig5-graph', style = {'flex': 1}),
            ],
            style = {'display': 'flex', 'gap': '10px'}
        ),
    ]
)

# One callback links the shared controls to all four figures:
# countries and the year range window fig1/fig2/fig5; fig3 uses the range's end year.
@app.callback(
    Output('fig1-graph', 'figure'),
    Output('fig2-graph', 'figure'),
    Output('fig3-graph', 'figure'),
    Output('fig5-graph', 'figure'),
    Input('country-dropdown', 'value'),
    Input('year-range', 'value'),
)
def update_figures(countries, year_range):
    if not countries:
        countries = default_countries
    year_range = sorted(year_range)
    years = range(year_range[0], year_range[1] + 1)  # full range for the time series
    year = year_range[1]                             # end year for the map
    return (
        make_fig1(countries, years),
        make_fig2(countries, years),
        make_fig3(year),
        make_fig5(countries, years),  # slope chart spans the range's start and end years
    )

# Serve the interactive version on port 8051 (the static Step 2 app is on 8050)
# make_server gives a shutdown handle, so re-running this cell replaces the server cleanly
from werkzeug.serving import make_server
if 'dash_server' in globals() and dash_server is not None:
    dash_server.shutdown()
dash_server = make_server('0.0.0.0', 8051, app, threaded = True)
Thread(target = dash_server.serve_forever, daemon = True).start()
print('Interactive dashboard serving at http://127.0.0.1:8051')

### Exploration narrative

*<The questions you started with, what you tried, dead ends, and what interaction revealed that a static chart would have missed — e.g. an outlier that turned out to belong to one category, a filter that reversed an initial impression.>*

## AI log and troubleshooting (10 pts)

Entry 1: Migrating off deprecated Dash imports
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: scaffold the first Dash app inside a Databricks notebook
- What came back / what happened: sample code importing `dash_core_components` and `dash_html_components`
- What was wrong or could be better: those packages were removed in Dash 2.x — the modern API is a single `dash` package
- How I noticed: the import failed when the cell ran
- What I did instead: switched to `from dash import Dash, dcc, html` and re-ran; imports succeeded

Entry 2: Outdated column reference broke every chart after cleaning changes
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: fix the local dashboard after I restricted `df` to only the columns I use and dropped rows without ISO codes
- What came back / what happened: every control change failed with `ValueError: Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['country', 'iso_code', 'year', …, 'energy generation', …] but received: total_energy_per_capita_kwh`
- What was wrong or could be better: the app cells still referenced the old column after the cleaned dataframe's schema changed — the whole dashboard depends on `df`'s schema
- How I noticed: the callback error message printed the available columns, which made the mismatch obvious
- What I did instead: updated the figure builder to plot the renamed `energy generation` column (and made the same fix in the deployed `app.py` so both stayed in sync)

Entry 3: RangeSlider rejected `allow_cross`
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: convert the single-year slider into a year-range control
- What came back / what happened: code using `allow_cross=False`
- What was wrong or could be better: the installed Dash version (dcc 4.5.0) expects the camelCase `allowCross` keyword
- How I noticed: `TypeError: The dcc.RangeSlider component (version 4.5.0) … received an unexpected keyword argument: allow_cross` — the error helpfully listed the accepted argument names
- What I did instead: renamed it to `allowCross=False` and re-ran successfully

Entry 4: Row 1 charts didn't span the full row width
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: put fig1 and fig2 side by side, spanning the row like the full-width charts below them
- What came back / what happened: the two graphs rendered narrow, sized to their content
- What was wrong or could be better: inside a `display: flex` container, children without a `flex` value don't stretch to share the row
- How I noticed: compared row 1 against the full-width fig3/fig4 rows in the running dashboard
- What I did instead: added `style={'flex': 1}` to both Graph components so they split the row evenly

Entry 5: Assistant inserted a new cell under the wrong heading
- Tool: Databricks Assistant (Genie Code)
- What I asked for / what I was trying to do: add the linked-interactivity code under the "Step 3" heading
- What came back / what happened: the cell was inserted after the "Step 2 — Layout" markdown instead
- What was wrong or could be better: the assistant referenced the wrong markdown cell's id when placing the new cell
- How I noticed: checked the notebook's cell order after the edit
- What I did instead: deleted the misplaced cell and re-inserted it under the correct "Step 3 — Linked interactivity" heading, then re-ran it

## Deployment instructions (10 pts)

The dashboard is deployed as a **Databricks App** (serverless hosting inside the workspace — no VM or server to manage).

**Live URL:** https://siads-521-energy-dashboard-7474654444631410.aws.databricksapps.com

*Note: the app is OAuth-protected, so you must be logged in to a Databricks account (any workspace member with access) to view it — anonymous visitors are redirected to the login page.*

### How it runs

The app's source lives in a workspace folder, `/Users/ncpeot@umich.edu/siads-521-energy-dashboard/`, containing four files:

* `app.py` — the full dashboard: data loading/cleaning (same steps as this notebook), the four figure builders, the layout, and the shared country/year-range callback. It exposes `server = app.server` as the web entrypoint.
* `app.yaml` — the run command: `gunicorn app:server --bind=0.0.0.0:8000` (gunicorn serves the Flask app that Dash wraps, on the port Databricks Apps expects)
* `requirements.txt` — `dash`, `pandas`, `plotly`, `gunicorn` (Databricks builds a fresh environment from this on every deploy)
* `owid_energy.csv` — the dataset, bundled with the app so it is self-contained

### Steps to repeat it

1. Create a workspace folder with the four files above (`app.py`, `app.yaml`, `requirements.txt`, and the CSV next to it — the app reads the CSV by relative path).
2. In the workspace sidebar, go to **Apps → Create app** (or `databricks apps create <app-name>`); app names must be lowercase alphanumeric with hyphens.
3. Deploy from the CLI (needs the Databricks CLI configured):
   `databricks apps deploy siads-521-energy-dashboard --source-code-path /Workspace/Users/<you>/siads-521-energy-dashboard`
   The app must be in RUNNING state before deploying — check with `databricks apps get <app-name>` and start it first if needed.
4. Open the app URL from `apps get` (the `url` field) in a browser where you're logged in to Databricks — the dashboard renders with the same linked interactivity as the notebook version.